In [0]:
!pip install boto3 wfdb matplotlib 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.5.0
    Not uninstalling fsspec at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-34c3510e-14de-43fa-aaac-e788cd435702
    Can't uninstall 'fsspec'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import scipy.signal as signal
import wfdb
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import shutil
import tempfile
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from rich import print
from rich.console import Console
from rich.table import Table

# --- 0. CONFIGURACIÓN ESTRUCTURAL ---
console = Console()
TARGET_FS = 250
WINDOW_BACK = int(TARGET_FS * 0.3)
WINDOW_FORWARD = int(TARGET_FS * 0.5)
BEAT_LENGTH = WINDOW_BACK + WINDOW_FORWARD

# 12 Derivaciones estándar estrictas
TARGET_LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
TARGET_LEADS_UPPER = [l.upper() for l in TARGET_LEADS]

# UNIFICACIÓN TOTAL EN /WORKSPACE
OUTPUT_DIR = '/Workspace/tmp/ecg_pvc_expert'
OUTPUT_PDF = f'{OUTPUT_DIR}/reporte_ejemplos_pvc.pdf'
S3_DEST = 's3://shazam-proyecto-ecg/silver_12leads/clase_5_pvc/'

# Variables de estado
global_signals = [] 
global_metadata = [] 
_lock = threading.Lock()
MAX_WORKERS = 12

# --- 1. PROCESAMIENTO CIENTÍFICO ---

def process_ecg_v1(raw, fs):
    """Resampling y filtrado estable. 
    Se usa sustracción de mediana para preservar amplitudes clínicas reales."""
    if fs != TARGET_FS:
        num_samples = int(len(raw) * (TARGET_FS / fs))
        raw = signal.resample(raw, num_samples)
    
    # Filtro 0.5 - 40 Hz
    nyq = 0.5 * TARGET_FS
    b, a = signal.butter(5, [0.5 / nyq, 40.0 / nyq], btype='band')
    filtered = signal.filtfilt(b, a, raw)
    
    # Normalización suave: centrado basal
    return filtered - np.median(filtered)

# --- 2. MOTOR DE INGESTA (INCART + ALINEACIÓN DE ENERGÍA GLOBAL) ---

def ingest_record(path, rec_id, pat_id, dataset, client, bucket):
    tmp_dir = tempfile.mkdtemp(dir='/Workspace/tmp')
    local_base = os.path.join(tmp_dir, rec_id)
    try:
        # Descargamos los archivos, incluyendo las anotaciones (.atr) Ground Truth
        for ext in ['.hea', '.dat', '.atr']:
            client.download_file(bucket, f"{path}{ext}", f"{local_base}{ext}")
        
        rec = wfdb.rdrecord(local_base)
        ann = wfdb.rdann(local_base, 'atr')
        names = [s.upper() for s in rec.sig_name]
        
        # GATEKEEPER DE LEADS: 12 derivaciones estrictas
        idxs = []
        try:
            for tl in TARGET_LEADS_UPPER:
                if tl == 'II' and 'II' not in names and 'MLII' in names:
                    idxs.append(names.index('MLII'))
                else:
                    idxs.append(names.index(tl))
        except ValueError: 
            return # Descartar si faltan leads

        signals = np.array([process_ecg_v1(rec.p_signal[:, i], rec.fs) for i in idxs])
        
        # Corrección de muestreo de INCART (257 Hz -> 250 Hz)
        resample_factor = TARGET_FS / rec.fs
        
        # GROUND TRUTH: Solo latidos Premature Ventricular Contraction ('V')
        pvc_raw_samples = [samp for samp, sym in zip(ann.sample, ann.symbol) if sym == 'V']

        count = 0
        ALIGN_W = int(TARGET_FS * 0.15) # ±150ms de búsqueda de energía

        for raw_samp in pvc_raw_samples:
            p_raw = int(raw_samp * resample_factor)
            
            # --- ALINEACIÓN FINA MULTI-DERIVACIÓN (CENTRADO PERFECTO) ---
            if p_raw - ALIGN_W >= 0 and p_raw + ALIGN_W < signals.shape[1]:
                
                # Sumamos la energía absoluta de las 12 derivaciones
                energy_curve = np.sum(np.abs(signals[:, p_raw - ALIGN_W : p_raw + ALIGN_W]), axis=0)
                local_shift = np.argmax(energy_curve) - ALIGN_W
                p_centrado = p_raw + local_shift
                
                # Extracción de ventana
                if p_centrado - WINDOW_BACK >= 0 and p_centrado + WINDOW_FORWARD < signals.shape[1]:
                    beat = signals[:, p_centrado - WINDOW_BACK : p_centrado + WINDOW_FORWARD]
                    
                    with _lock:
                        global_signals.append(beat.astype(np.float32))
                        global_metadata.append({
                            'patient_id': pat_id, 
                            'record_id': rec_id,
                            'dataset': dataset, 
                            'beat_id': len(global_signals) - 1,
                            'class_name': 'PVC',
                            'label': 2, # Asumiendo que 2 es la clase para PVC en tu pipeline general
                            'label_quality': 'cardiologist'
                        })
                        count += 1
                            
        if count > 0:
            print(f"[bold green][OK][/bold green] {dataset} | Pat: {pat_id} | PVCs Extraídos y Centrados: {count}")

    except Exception as e:
        pass
    finally: shutil.rmtree(tmp_dir, ignore_errors=True)

# --- 3. REPORTE VISUAL ---

def generate_report_pdf(X, meta_df, output_path):
    print(f"\n[yellow]Generando PDF de validación visual (12 Derivaciones PVC Ground Truth): {output_path}[/yellow]")
    
    with PdfPages(output_path) as pdf:
        t = np.linspace(-WINDOW_BACK/TARGET_FS, WINDOW_FORWARD/TARGET_FS, BEAT_LENGTH, endpoint=False)
        
        for ds in meta_df['dataset'].unique():
            ds_meta = meta_df[meta_df['dataset'] == ds].drop_duplicates(subset=['patient_id']).head(25)
            if ds_meta.empty: continue

            for i, (_, row) in enumerate(ds_meta.iterrows()):
                beat = X[row['beat_id']]
                
                fig, axs = plt.subplots(4, 3, figsize=(18, 16), sharex=True)
                fig.suptitle(f"Clase 2: {ds.upper()} - Auditoría PVC (Ground Truth)\nPaciente: {row['patient_id']} | Registro: {row['record_id']}", 
                             fontsize=18, fontweight='bold', y=0.96)
                
                axs = axs.flatten()
                for ch in range(12):
                    ax = axs[ch]
                    ax.plot(t, beat[ch], color='#1f77b4', linewidth=1.5)
                    
                    # Grilla Milimétrica
                    ax.grid(True, which='both', color='red', linestyle='--', linewidth=0.5, alpha=0.3)
                    ax.minorticks_on()
                    ax.grid(True, which='minor', color='red', linestyle=':', linewidth=0.2, alpha=0.2)
                    
                    # Línea negra en 0.0s exactos: Centro de Energía PVC
                    ax.axvline(0, color='black', linewidth=1.5, linestyle='-') 
                    ax.axvspan(-0.06, 0.20, color='orange', alpha=0.15, label='Centro de Energía PVC') 
                    
                    ax.set_title(f"Derivación {TARGET_LEADS[ch]}", fontsize=12, fontweight='bold')
                    if ch == 0: ax.legend(loc='upper right', fontsize=8)
                    
                plt.tight_layout(rect=[0, 0.03, 1, 0.93])
                pdf.savefig(fig)
                plt.close(fig)

# --- 4. EJECUCIÓN PRINCIPAL ---

if __name__ == "__main__":
    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    s3_open = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
    tasks = []

    print("[bold cyan]--- INICIANDO INGESTA CLASE 2 (PVC - GROUND TRUTH INCART) ---[/bold cyan]")
    for i in range(1, 76):
        rid = f"I{i:02d}"
        tasks.append((ingest_record, f"incartdb/1.0.0/{rid}", rid, f"INCART_{rid}", "INCART", s3_open, 'physionet-open'))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as exe:
        list(exe.map(lambda p: p[0](*p[1:]), tasks))

    # --- 5. SPLIT Y GUARDADO EN CARPETAS (TRAIN / VAL / TEST) ---
    if len(global_signals) > 0:
        X_all = np.array(global_signals)
        meta_df = pd.DataFrame(global_metadata)
        
        print("\n[bold cyan]--- PROCESAMIENTO FINALIZADO. APLICANDO SPLIT CLÍNICO ---[/bold cyan]")
        
        # SPLIT POR PACIENTE
        pats = meta_df['patient_id'].unique()
        if len(pats) >= 3:
            train_p, test_p = train_test_split(pats, test_size=0.2, random_state=42)
            if len(test_p) > 1:
                val_p, test_p = train_test_split(test_p, test_size=0.5, random_state=42)
            else:
                val_p = [] 
        else:
            train_p, val_p, test_p = pats, [], []

        def set_split(p):
            if p in train_p: return 'train'
            if p in val_p: return 'val'
            return 'test'
            
        meta_df['split'] = meta_df['patient_id'].apply(set_split)

        # GUARDADO EN SUBCARPETAS
        for s in ['train', 'val', 'test']:
            subset = meta_df[meta_df['split'] == s]
            if subset.empty: continue
                
            idx_list = subset['beat_id'].tolist()
            X_sub = X_all[idx_list]
            
            split_dir = os.path.join(OUTPUT_DIR, s)
            os.makedirs(split_dir, exist_ok=True)
            
            # Guardamos las señales y la metadata
            np.savez_compressed(f"{split_dir}/pvc_signals.npz", x=X_sub)
            
            # Limpiamos columnas internas antes de guardar (quitamos 'beat_id' y 'split')
            export_meta = subset.drop(columns=['beat_id', 'split'])
            export_meta.to_csv(f"{split_dir}/pvc_meta.csv", index=False)
            print(f"[bold green]💾 Exportado {s}: {X_sub.shape[0]} latidos en '{s}/'[/bold green]")

        # Generar PDF de validación global
        generate_report_pdf(X_all, meta_df, OUTPUT_PDF)

        # Tabla Resumen
        table = Table(title="Auditoría Final de Ingesta PVC (INCART)")
        table.add_column("Dataset", justify="left", style="cyan")
        table.add_column("Total Beats", justify="right", style="green")
        table.add_column("Pacientes", justify="right", style="magenta")

        for ds in meta_df['dataset'].unique():
            beats_total = len(meta_df[meta_df['dataset'] == ds])
            pats_total = meta_df[meta_df['dataset'] == ds]['patient_id'].nunique()
            table.add_row(ds, str(beats_total), str(pats_total))
        console.print(table)

        # --- 6. SUBIDA ESTRUCTURADA A S3 (DATABRICKS dbutils) ---
        try:
            print(f"\n[cyan]Subiendo a {S3_DEST}...[/cyan]")
            for split_folder in ['train', 'val', 'test']:
                local_split_dir = os.path.join(OUTPUT_DIR, split_folder)
                if os.path.exists(local_split_dir):
                    s3_split_dest = f"{S3_DEST}{split_folder}/" 
                    for f in os.listdir(local_split_dir):
                        dbutils.fs.cp(f"file:{os.path.join(local_split_dir, f)}", f"{s3_split_dest}{f}")
                        
            if os.path.exists(OUTPUT_PDF):
                dbutils.fs.cp(f"file:{OUTPUT_PDF}", f"{S3_DEST}{os.path.basename(OUTPUT_PDF)}")
                print(f"[bold green]✅ PDF subido a {S3_DEST}{os.path.basename(OUTPUT_PDF)}[/bold green]")
                
        except NameError:
            print("[yellow]dbutils no está definido. Saltando subida a S3 (ejecución local detectada).[/yellow]")

    else:
        print("[bold red]ERROR: No se detectaron anotaciones 'V' (PVC) válidas en INCART.[/bold red]")

--- INICIANDO INGESTA CLASE 2 (PVC - GROUND TRUTH INCART) ---

[OK] INCART | Pat: INCART_I03 | PVCs Extraídos y Centrados: 125

[OK] INCART | Pat: INCART_I05 | PVCs Extraídos y Centrados: 247

[OK] INCART | Pat: INCART_I01 | PVCs Extraídos y Centrados: 344

[OK] INCART | Pat: INCART_I04 | PVCs Extraídos y Centrados: 121

...

💾 Exportado test: 2100 latidos en 'test/'

Generando PDF de validación visual (12 Derivaciones PVC Ground Truth): 
/Workspace/tmp/ecg_pvc_expert/reporte_ejemplos_pvc.pdf

   Auditoría Final de Ingesta PVC    
              (INCART)               
┏━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Dataset ┃ Total Beats ┃ Pacientes ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ INCART  │       20007 │        70 │
└─────────┴─────────────┴───────────┘

Subiendo a s3://shazam-proyecto-ecg/silver_12leads/clase_5_pvc/...

✅ PDF subido a s3://shazam-proyecto-ecg/silver_12leads/clase_5_pvc/reporte_ejemplos_pvc.pdf